In [111]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [112]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [113]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()
openai_client = OpenAI(api_key=os.getenv("SECRET_OPENAI_API_KEY"))

In [114]:
import json
from evaluation_utils import llm_structured_retry
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]


def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "filename": doc["filename"]
        })

    return results, usage

In [115]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [123]:
usages

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=110, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1130),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=107, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1393),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=1280, cache_write_tokens=0), output_tokens=120, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1873)]

In [116]:
import pandas as pd

ground_truth_df = pd.read_csv("data/ground-truth.csv")

ground_truth_df.head()

ground_truth = ground_truth_df.to_dict(orient="records")

In [117]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [118]:
from minsearch import Index, VectorSearch
from embedder import Embedder
import numpy as np

model = Embedder()

vectors = []

for chunk in chunks:
    content_embedded = model.encode(chunk['content'])
    vectors.append(content_embedded)

X = np.array(vectors)


text_search=Index(text_fields=["filename"])
vector_search=VectorSearch(keyword_fields=["filename"])

text_search.fit(chunks)
vector_search.fit(X, chunks)

In [119]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [120]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [121]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [122]:
text_search.search(q, num_results=5)

[{'start': 0,
  'content': '# Generating RAG Answers\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=utkcclfpj0g&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the first part of this module, we evaluated search quality. We\nchecked whether the right document appeared in the search results.\n\nNow we evaluate the full RAG pipeline. For each generated question, we\nrun RAG and save the answer produced by the LLM. Later, we\'ll compare\nthis answer with the original FAQ answer.\n\nThis is the A->Q->A\' setup:\n\n- A = original answer in the FAQ\n- Q = generated question from this answer\n- A\' = answer produced by our RAG system\n\nIf A\' is close to A, the RAG system is doing a good job.\n\nThis is still offline evaluation. We can compare A and A\' because our\nquestions came from FAQ records. For each question, we know which\noriginal answer it came from.\n\n## Loading the data\n\nCreate a new notebook for RAG evaluation.\n\nLoad the ground truth questions:\n\n```python\ni

In [125]:
qv = model.encode(q)

vector_search.search(qv, num_results=5)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [127]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [128]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [130]:
relevance_total_text = compute_relevance_total_text(ground_truth)

  0%|          | 0/360 [00:00<?, ?it/s]

KeyError: 'document'